In [0]:
amostra_clientes = (
    spark.table("credit_risk_pipeline.bronze.application")
    .select("SK_ID_CURR")
    .sample(fraction=0.13, seed=42)
)

amostra_clientes.write.mode("overwrite").saveAsTable("credit_risk_pipeline.silver.clientes_amostra")

print(f"Total de clientes na amostra: {amostra_clientes.count()}")

In [0]:
from pyspark.sql import functions as F

df_completa = spark.table("credit_risk_pipeline.bronze.application")
taxa_completa = df_completa.agg(F.avg("TARGET")).collect()[0][0]

df_amostra = df_completa.join(amostra_clientes, on="SK_ID_CURR", how="inner")
taxa_amostra = df_amostra.agg(F.avg("TARGET")).collect()[0][0]

print(f"Taxa de inadimplência (base completa): {taxa_completa:.4f}")
print(f"Taxa de inadimplência (amostra): {taxa_amostra:.4f}")

### Silver: Application

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("credit_risk_pipeline.bronze.application")
amostra = spark.table("credit_risk_pipeline.silver.clientes_amostra")

colunas_relevantes = [
    "SK_ID_CURR", "TARGET", "NAME_CONTRACT_TYPE", "CODE_GENDER",
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE", "DAYS_BIRTH", "DAYS_EMPLOYED", "OCCUPATION_TYPE",
    "CNT_FAM_MEMBERS", "REGION_RATING_CLIENT", "ORGANIZATION_TYPE",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"
]

df_silver = (
    df_bronze
    .join(amostra, on="SK_ID_CURR", how="inner")
    .select(colunas_relevantes)
    .replace("XNA", None)
    .withColumn("idade_anos", (F.col("DAYS_BIRTH") * -1 / 365).cast("int"))
    .withColumn(
        "anos_empregado",
        F.when(F.col("DAYS_EMPLOYED") == 365243, None)
         .otherwise((F.col("DAYS_EMPLOYED") * -1 / 365).cast("int"))
    )
    .drop("DAYS_BIRTH", "DAYS_EMPLOYED")
)

df_silver.write.mode("overwrite").saveAsTable("credit_risk_pipeline.silver.application")

print(f"Tabela silver.application criada com {df_silver.count()} linhas e {len(df_silver.columns)} colunas.")

### Silver: Bureau

In [0]:
df_bronze_bureau = spark.table("credit_risk_pipeline.bronze.bureau")
amostra = spark.table("credit_risk_pipeline.silver.clientes_amostra")

colunas_relevantes_bureau = [
    "SK_ID_BUREAU", "SK_ID_CURR", "CREDIT_ACTIVE", "DAYS_CREDIT",
    "CREDIT_DAY_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_MAX_OVERDUE", "CREDIT_TYPE"
]

df_silver_bureau = (
    df_bronze_bureau
    .join(amostra, on="SK_ID_CURR", how="inner")
    .select(colunas_relevantes_bureau)
    .fillna({
        "AMT_CREDIT_SUM_DEBT": 0,
        "AMT_CREDIT_SUM_OVERDUE": 0,
        "AMT_CREDIT_MAX_OVERDUE": 0
    })
)

df_silver_bureau.write.mode("overwrite").saveAsTable("credit_risk_pipeline.silver.bureau")

print(f"Tabela silver.bureau criada com {df_silver_bureau.count()} linhas e {len(df_silver_bureau.columns)} colunas.")

### Silver: Previous Application

In [0]:
df_bronze_previous = spark.table("credit_risk_pipeline.bronze.previous_application")
amostra = spark.table("credit_risk_pipeline.silver.clientes_amostra")

colunas_relevantes_previous = [
    "SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_TYPE", "AMT_APPLICATION",
    "AMT_CREDIT", "AMT_ANNUITY", "NAME_CONTRACT_STATUS", "DAYS_DECISION",
    "NAME_CASH_LOAN_PURPOSE", "CODE_REJECT_REASON", "CNT_PAYMENT"
]

df_silver_previous = (
    df_bronze_previous
    .join(amostra, on="SK_ID_CURR", how="inner")
    .select(colunas_relevantes_previous)
    .replace("XNA", None)
)

df_silver_previous.write.mode("overwrite").saveAsTable("credit_risk_pipeline.silver.previous_application")

print(f"Tabela silver.previous_application criada com {df_silver_previous.count()} linhas e {len(df_silver_previous.columns)} colunas.")

### Silver: Installments Payments

In [0]:
df_bronze_installments = spark.table("credit_risk_pipeline.bronze.installments_payments")
amostra = spark.table("credit_risk_pipeline.silver.clientes_amostra")

colunas_relevantes_installments = [
    "SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_NUMBER",
    "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"
]

df_silver_installments = (
    df_bronze_installments
    .join(amostra, on="SK_ID_CURR", how="inner")
    .select(colunas_relevantes_installments)
    .withColumn("parcela_paga", F.col("AMT_PAYMENT").isNotNull())
    .withColumn("atraso_dias", F.col("DAYS_ENTRY_PAYMENT") - F.col("DAYS_INSTALMENT"))
    .withColumn("valor_nao_pago", F.col("AMT_INSTALMENT") - F.col("AMT_PAYMENT"))
)

df_silver_installments.write.mode("overwrite").saveAsTable("credit_risk_pipeline.silver.installments_payments")

print(f"Tabela silver.installments_payments criada com {df_silver_installments.count()} linhas e {len(df_silver_installments.columns)} colunas.")